# 아래 윤곽을 가린 원판을 두 가지 정답으로 평가하기

08번 실험의 검증 31장에 아래 끝 가림과 바깥 연장을 각각 적용합니다. 합성 전 정답과 합성 후 보이는 원판의 정답에 똑같은 탐지 후보를 대조합니다. 신뢰도 0.10, 매칭 IoU 0.50을 유지합니다. 원판이 뒤에 있어도 보이지 않는 부분을 추정해서 포함하는지에 따라 평가 결과가 달라지는지 확인하는 실험입니다.

평가용 박스 62개는 모델 예측을 표시하지 않은 원본·합성 쌍을 확대해 한 번 판독했습니다. 가까운 원판에서 식별되는 부분을 모두 포함하는 직사각형으로 그렸으며, 패치 양옆에 남은 테두리와 패치 아래로 보이는 끝도 포함합니다. 그래서 박스 안에 패치 일부가 들어갈 수 있습니다. a03의 2번은 합성 전부터 가려져 기존 라벨도 부분 박스입니다. 이 사례는 양쪽 판정에서 따로 확인합니다.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Rectangle
from ultralytics import YOLO

from wlcv.config import ANNOTATIONS, DATASET, RUNS
from wlcv.evaluation.patch_position import make_example, paste_patch, reviewed_floor_frames
from wlcv.evaluation.visible_box import COORDS, compare_visible_boxes, summarize_comparison

runs = {"Baseline": "plate_baseline", "Texture": "plate_occlusion_texture"}
report = RUNS / "plate_baseline/report"
floors = reviewed_floor_frames(report / "occlusion_half_area/mask_positions.csv")
visible_csv = ANNOTATIONS / "visible_plate_boxes.csv"
visible = pd.read_csv(visible_csv).set_index(["stem", "condition"])
out = report / "visible_box_comparison"
out.mkdir(parents=True, exist_ok=True)

def draw_box(axis, rect, color, style, label):
    x, y, x2, y2 = rect
    axis.add_patch(Rectangle((x,y), x2-x, y2-y, fill=False, edgecolor=color,
                             linewidth=2, linestyle=style, label=label))

## 1. 예측을 숨기고 원판 윤곽과 평가 박스 확인

하늘색 실선은 합성 전 정답, 노란 점선은 현재 보이는 원판을 판독한 박스입니다. 예시 프레임을 바꾸며 패치 아래나 양옆에 남은 부분도 확인할 수 있습니다.

In [ ]:
stem = "b03_sn_f_01_f8"
image, original, _, donor, patches = make_example(DATASET, floors, stem)
fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.8), layout="constrained")
for axis, condition in zip(axes, ["Lower strip", "Cross lower edge"]):
    axis.imshow(paste_patch(image, patches[condition]))
    draw_box(axis, original, "#44e0ff", "-", "Pre-mask")
    draw_box(axis, visible.loc[(stem, condition), COORDS], "#ffca40", "--", "Visible")
    x, y, r, b = original
    axis.set(xlim=(x-(r-x)*0.35, r+(r-x)*0.35), ylim=(b+(b-y)*0.45, y-(b-y)*0.25),
             title=condition)
    axis.axis("off")
axes[0].legend(loc="lower left", fontsize=8)
plt.show()

## 2. 입력 한 번당 후보 한 목록으로 평가

Mac CPU, 입력 크기 640, 추론 하한 0.001을 사용합니다. 같은 모델·프레임·조건 안에서 후보 번호와 신뢰도는 고정됩니다. 각 후보에 두 IoU를 계산한 뒤, 신뢰도와 IoU 기준을 동시에 통과하는 후보가 한 개라도 있으면 정탐입니다. 패치와 합성 전 기준 점수를 지난 실험과 대조합니다.

In [ ]:
previous = pd.read_csv(report / "patch_position_comparison/per_frame.csv")
frame_results, predictions = [], []
for name, run in runs.items():
    trained = pd.read_csv(RUNS / run / "train_people.csv")
    assert set(floors.person).isdisjoint(trained.person_id)
    model = YOLO(str((RUNS / run / "weights/best.pt").resolve(strict=True)))
    assert model.names == {0: "plate"}
    rows, boxes = compare_visible_boxes(model, DATASET, floors, visible_csv)
    keys = ["stem", "condition"]
    old = previous[previous.model.eq(name) & previous.condition.isin(rows.condition)].set_index(keys).sort_index()
    new = rows.set_index(keys).sort_index()
    for col in ["donor", "x", "y", "width", "height", "patch_hash"]:
        pd.testing.assert_series_equal(new[col], old[col], check_dtype=False)
    for after, before in [("original_hit", "detected"), ("original_score", "confidence"), ("original_iou", "iou")]:
        pd.testing.assert_series_equal(new[after], old[before], check_names=False, atol=1e-6, rtol=0)
    frame_results.append(rows.assign(model=name))
    predictions.append(boxes.assign(model=name))

decisions = pd.concat(frame_results, ignore_index=True)
candidates = pd.concat(predictions, ignore_index=True)
summary = summarize_comparison(decisions)
decisions.to_csv(out / "per_frame.csv", index=False)
candidates.to_csv(out / "candidates.csv", index=False)
summary.to_csv(out / "summary.csv")
display(summary.round(3))

## 3. 정탐 수와 IoU 변화를 따로 읽기

`gained`와 `lost`는 정답 기준만 바꿨을 때 새로 잡거나 놓친 장수입니다. 미탐 중 `low_score_match`는 0.001 이상에서 어느 한 정답에라도 위치가 맞았지만 신뢰도 0.10에 미달한 경우입니다. `no_match_at_0001`은 두 정답에 모두 맞는 후보를 각각 찾아봤을 때 어느 쪽에도 매칭 후보가 없는 경우입니다.

그림에서는 신뢰도 0.10을 넘고 어느 한 정답에라도 매칭된 후보 중 최고 신뢰도 하나를 고정합니다. 가로·세로 좌표는 같은 후보의 IoU이며, 점선 아래에서는 보이는 부분 기준 IoU가 더 작습니다. 신뢰도 기준을 넘은 후보가 없는 입력은 표에서 미탐으로 집계합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.7, 4), sharex=True, sharey=True, layout="constrained")
for axis, name in zip(axes, runs):
    for condition, marker, color in [("Lower strip", "o", "#176785"), ("Cross lower edge", "^", "#b45325")]:
        group = decisions[(decisions.model == name) & (decisions.condition == condition)].dropna(subset=["fixed_candidate"])
        axis.scatter(group.fixed_original_iou, group.fixed_visible_iou, marker=marker, color=color,
                     alpha=0.75, s=30, label=f"{condition} (n={len(group)})")
    axis.plot([0.45,1], [0.45,1], "--", color="#6e7378", lw=1)
    axis.axhline(0.50, color="gray", linestyle=":")
    axis.axvline(0.50, color="gray", linestyle=":")
    axis.set(xlim=(0.45,1), ylim=(0.45,1), xlabel="IoU with pre-mask box", title=name, aspect="equal")
    axis.legend(loc="upper left", fontsize=8)
axes[0].set_ylabel("IoU with visible-part box")
fig.savefig(out / "same_candidate_iou.png", dpi=150)
plt.show()
missed = decisions[~decisions.original_hit & ~decisions.visible_hit]
display(missed.groupby(["model", "condition", "above_threshold"]).size().rename("missed_frames"))
display(summarize_comparison(decisions[decisions.stem != "a03_sn_f_01_f2"]).round(3))

## 4. 낮은 신뢰도 후보와 실제 검출의 박스 비교

a03 19번은 저신뢰 예시이고 b03 8번은 검출 예시입니다. 각 칸에서 합성 전 정답에 매칭된 최고 신뢰도 후보 하나를 골라 보라색으로 그립니다. 표기한 두 IoU와 신뢰도는 모두 그 동일한 후보에서 계산합니다.

In [ ]:
examples = ["a03_sn_f_01_f19", "b03_sn_f_01_f8"]
condition = "Cross lower edge"
fig, axes = plt.subplots(2, 2, figsize=(8.3, 6.7), layout="constrained")
indexed = decisions.set_index(["model", "stem", "condition"])
for row, stem in enumerate(examples):
    image, original, _, _, patches = make_example(DATASET, floors, stem)
    for axis, name in zip(axes[row], runs):
        axis.imshow(paste_patch(image, patches[condition]))
        draw_box(axis, original, "#44e0ff", "-", "Pre-mask")
        draw_box(axis, visible.loc[(stem, condition), COORDS], "#ffca40", "--", "Visible")
        candidate = indexed.loc[(name, stem, condition), "original_candidate"]
        subtitle = "No match to pre-mask box"
        if pd.notna(candidate):
            box = candidates[(candidates.model == name) & (candidates.stem == stem) &
                             (candidates.condition == condition) & (candidates.candidate == candidate)].iloc[0]
            draw_box(axis, box[COORDS], "#e279eb", "-.", "Prediction")
            subtitle = f"score {box.confidence:.3f}   IoU {box.original_iou:.2f} → {box.visible_iou:.2f}"
        x, y, r, b = original
        axis.set(xlim=(x-(r-x)*0.4,r+(r-x)*0.4), ylim=(b+(b-y)*0.45,y-(b-y)*0.3),
                 title=f"{stem} · {name}\n{subtitle}")
        axis.axis("off")
handles, names = axes[0,0].get_legend_handles_labels()
fig.legend(handles, names, loc="outside lower center", ncol=3, frameon=False)
fig.savefig(out / "two_reference_examples.png", dpi=150)
plt.show()